[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/cyneuro/SCP/blob/main/7_tools.ipynb)

# SCP Step 7 - Extra Tools

Optional notebook access to small utility scripts for users who prefer notebooks over terminal commands.

Current tools:
1. Restore saved run values into tune configs.
2. Export `spikes.npz` to a trial-by-trial CSV.
3. Merge two compatible run outputs.
4. Clear top-level `slurm_*` output folders.

All write/delete operations are gated by explicit `RUN_* = True` toggles. Defaults are dry-run or preview mode.


## 7.0 Environment Setup

Run this first. It finds or clones SCP, configures imports, and installs notebook dependencies in Colab when requested.

Quick guide:
- Leave `SCP_REPO_URL`, `SCP_REPO_BRANCH`, `SCP_REPO_DIR`, and `INSTALL_DEPS` unchanged for normal local use.
- In Colab, the cell can clone the repo and install dependencies.
- For private repos, set `SCP_GIT_TOKEN`, `SCP_GITHUB_TOKEN`, or `GITHUB_TOKEN` before running.


In [ ]:
# Environment setup: works locally or in Google Colab
%load_ext autoreload
%autoreload 2

import os
import subprocess
import sys
from pathlib import Path

# User-editable only when running in a fresh Colab or unusual local layout.
SCP_REPO_URL = os.environ.get("SCP_REPO_URL", "https://github.com/cyneuro/SCP.git")
SCP_REPO_BRANCH = os.environ.get("SCP_REPO_BRANCH", "") or None
IN_COLAB = "COLAB_RELEASE_TAG" in os.environ
AUTO_CLONE_REPO = os.environ.get("SCP_AUTO_CLONE", "1" if IN_COLAB else "0") not in {"0", "false", "False"}
INSTALL_DEPS = None  # None = install automatically in Colab, do not install locally.
SCP_REPO_DIR = Path(os.environ.get("SCP_REPO_DIR", "/content/SCP" if IN_COLAB else str(Path.cwd() / "SCP")))


def _looks_like_scp_repo(path: Path) -> bool:
    return (path / "modules").is_dir() and (path / "run_pipeline.py").is_file()


repo_root = None
env_root = os.environ.get("SCP_ROOT")
if env_root and _looks_like_scp_repo(Path(env_root).expanduser()):
    repo_root = Path(env_root).expanduser().resolve()
else:
    start = Path.cwd().resolve()
    candidates = list((start, *start.parents))
    for base in (start, start.parent):
        try:
            candidates.extend(child for child in base.iterdir() if child.is_dir())
        except Exception:
            pass
    for candidate in candidates:
        if _looks_like_scp_repo(candidate):
            repo_root = candidate.resolve()
            break

if repo_root is None:
    if not AUTO_CLONE_REPO:
        raise FileNotFoundError("Could not find SCP. Set SCP_ROOT or enable SCP_AUTO_CLONE=1.")
    clone_url = SCP_REPO_URL
    token = os.environ.get("SCP_GIT_TOKEN") or os.environ.get("SCP_GITHUB_TOKEN") or os.environ.get("GITHUB_TOKEN")
    if token and clone_url.startswith("https://") and "@" not in clone_url:
        clone_url = clone_url.replace("https://", f"https://{token}@", 1)
    clone_cmd = ["git", "clone", "--depth", "1"]
    if SCP_REPO_BRANCH:
        clone_cmd += ["--branch", SCP_REPO_BRANCH]
    clone_cmd += [clone_url, str(SCP_REPO_DIR)]
    subprocess.check_call(clone_cmd)
    repo_root = SCP_REPO_DIR.resolve()

os.environ["SCP_ROOT"] = str(repo_root)
if str(repo_root) not in sys.path:
    sys.path.insert(0, str(repo_root))

from modules.notebooks.bootstrap import ensure_notebook_dependencies, is_colab

ensure_notebook_dependencies(install_deps=INSTALL_DEPS)

print("Runtime:", "Colab" if is_colab() else "local")
print("SCP repo:", repo_root)


## 7.1 Restore Run State to Tune Configs

Restore **values only** from a saved run into a target tune while preserving the target file structure.

Typical use cases:
- recover config values from a saved run after manual edits,
- copy selected synapse-group values between related tune folders,
- inspect what would change before applying anything.

Quick guide:
- `RUN_RESTORE = False`: dry-run preview.
- `RUN_RESTORE = True`: write changes.
- `apply`: choose config classes to restore.
- `syn_groups`: `"all"` or a comma-separated subset such as `"pn_exc,bg_exc"`.
- `backup`: create timestamped backups before writes.

CLI equivalent:

```bash
python scripts/restore_run_state.py \
  --from-run cells/SST/tunes/tuned/output_data/example_run \
  --to-tune cells/SST/tunes/tuned \
  --apply sim_config,cell_config,geometry,syn_config,syn_groups \
  --syn-groups all

python scripts/restore_run_state.py \
  --from-run cells/SST/tunes/tuned/output_data/example_run \
  --to-tune cells/SST/tunes/tuned \
  --apply sim_config,syn_config,syn_groups,fit_json \
  --syn-groups pn_exc,bg_exc \
  --write
```


In [ ]:
from pathlib import Path
from scripts.restore_run_state import restore_run_state, print_report

# False -> dry-run preview, True -> apply changes
RUN_RESTORE = False

from_run = Path("cells/SST/tunes/tuned/output_data/example_run")
to_tune = Path("cells/SST/tunes/tuned")
apply = ["fit_json", "cell_config", "geometry", "syn_config", "syn_groups"]
syn_groups = "all"  # or e.g. "pn_exc,bg_exc,vip_inh"
allow_source_fallback = True
backup = True

report = restore_run_state(
    from_run=from_run,
    to_tune=to_tune,
    apply=apply,
    syn_groups=syn_groups,
    dry_run=not RUN_RESTORE,
    allow_source_fallback=allow_source_fallback,
    backup=backup,
)
print_report(report)


## 7.2 Export Spikes CSV

Export `spikes.npz` to one CSV row per trial.

Typical use cases:
- inspect spike times in a spreadsheet,
- share compact trial-level spike data,
- quickly compare runs outside Python.

Quick guide:
- `RUN_EXPORT_SPIKES = False`: preview source/output paths.
- `RUN_EXPORT_SPIKES = True`: write the CSV.
- `spikes_source`: run folder, `results/` folder, or direct `spikes.npz` path.
- `spikes_csv_out = None`: write beside `spikes.npz` as `spikes_trials.csv`.

CLI equivalent:

```bash
python scripts/export_spikes_csv.py \
  --input cells/PV/tunes/tuned/output_data/example_run_a \
  --delimiter "," \
  --precision 10 \
  --trial-prefix trial_
```


In [ ]:
from pathlib import Path
from modules.analysis import analysis

# False -> dry-run preview, True -> export CSV
RUN_EXPORT_SPIKES = False

# Set to one of: run folder, results folder, or direct spikes.npz path.
spikes_source = Path("cells/SST/tunes/tuned/output_data/example_run")
spikes_csv_out = None  # e.g. Path("cells/SST/tunes/tuned/output_data/example_run/results/my_spikes_trials.csv")
spikes_time_delimiter = ","
spikes_precision = 10
spikes_trial_prefix = "trial_"
spikes_overwrite = False

source = Path(spikes_source)
if source.is_dir():
    candidate = source / "results" / "spikes.npz"
    if not candidate.is_file():
        candidate = source / "spikes.npz"
else:
    candidate = source

if not candidate.is_file():
    raise FileNotFoundError(f"Could not find spikes.npz from source: {source}")

default_out = candidate.with_name("spikes_trials.csv")
resolved_out = Path(spikes_csv_out) if spikes_csv_out is not None else default_out

if not RUN_EXPORT_SPIKES:
    print("[dry-run] Would export spikes CSV with:")
    print(f"  source spikes: {candidate}")
    print(f"  out csv      : {resolved_out}")
    print(f"  delimiter    : {spikes_time_delimiter!r}")
    print(f"  precision    : {spikes_precision}")
    print(f"  trial_prefix : {spikes_trial_prefix!r}")
    print(f"  overwrite    : {spikes_overwrite}")
else:
    out_csv = analysis.export_spikes_trials_csv(
        source,
        out_csv=spikes_csv_out,
        delimiter=spikes_time_delimiter,
        precision=spikes_precision,
        overwrite=spikes_overwrite,
        trial_prefix=spikes_trial_prefix,
    )
    print(f"Saved spikes trial CSV: {out_csv}")


## 7.3 Merge Two Outputs

Merge two compatible run outputs into one multi-trial run with a config-first safety gate.

Typical use cases:
- combine split trial batches,
- merge compatible local/SLURM pieces,
- dry-run config comparisons before joining results.

Quick guide:
- `RUN_MERGE = False`: dry-run compatibility preview.
- `RUN_MERGE = True`: write the merged run.
- `strict_configs = True`: block merges when compared configs differ.
- `keep_logs = True`: save merge reports under the merged run's `logs/` folder.

Output layout on write:
- `<output_dir>/<output_stem>/results/...`
- `<output_dir>/<output_stem>/logs/merge_<timestamp>/...`

CLI equivalent:

```bash
python scripts/merge_two_runs.py \
  --run-a cells/PV/tunes/tuned/output_data/example_run_a \
  --run-b cells/PV/tunes/tuned/output_data/example_run_b

python scripts/merge_two_runs.py \
  --run-a cells/PV/tunes/tuned/output_data/example_run_a \
  --run-b cells/PV/tunes/tuned/output_data/example_run_b \
  --output-dir cells/PV/tunes/tuned/output_data \
  --output-stem merge_example \
  --write
```


In [ ]:
from pathlib import Path
from scripts.merge_two_runs import merge_two_runs, print_report

# False -> dry-run preview, True -> merge and write output
RUN_MERGE = False

run_a = Path("cells/PV/tunes/tuned/output_data/example_run_a")
run_b = Path("cells/PV/tunes/tuned/output_data/example_run_b")
output_dir = Path("cells/PV/tunes/tuned/output_data")
output_stem = "merge_example"  # run folder name under output_dir
strict_configs = True
keep_logs = True
max_diffs = 200

report = merge_two_runs(
    run_a=run_a,
    run_b=run_b,
    output_dir=output_dir,
    output_stem=output_stem,
    dry_run=not RUN_MERGE,
    strict_configs=strict_configs,
    keep_logs=keep_logs,
    max_diffs=max_diffs,
)
print_report(report)


## 7.4 Clear `slurm_*` Runs

Delete top-level runs in a tune's `output_data` whose names start with `slurm_`.

Typical use cases:
- clean failed or temporary SLURM batch folders,
- remove old array-run scratch outputs after merging,
- preview exactly what would be deleted before writing.

Quick guide:
- `RUN_CLEAR_SLURM = False`: dry-run preview.
- `RUN_CLEAR_SLURM = True`: delete matching entries.
- `slurm_prefix`: defaults to `"slurm_"`; change only if you intentionally use a different prefix.
- `output_data_dir`: optional direct override when not using `tune_dir`.

CLI equivalent:

```bash
python scripts/clear_slurm_runs.py \
  --tune-dir cells/SST/tunes/tuned

python scripts/clear_slurm_runs.py \
  --tune-dir cells/SST/tunes/tuned \
  --write
```


In [ ]:
from pathlib import Path
from scripts.clear_slurm_runs import clear_slurm_runs, print_report

# False -> dry-run preview, True -> delete matching runs
RUN_CLEAR_SLURM = False

tune_dir = Path("cells/PV/tunes/tuned")
output_data_dir = None  # optional direct override to output_data path
slurm_prefix = "slurm_"
max_items_to_show = 120

report = clear_slurm_runs(
    tune_dir=tune_dir if output_data_dir is None else None,
    output_data_dir=output_data_dir,
    prefix=slurm_prefix,
    dry_run=not RUN_CLEAR_SLURM,
)
print_report(report, max_items=max_items_to_show)


## Future Tool Candidates

Keep Step 7 limited to small, self-contained utilities. Good candidates are tools that already have a script or callable function and are useful to run without terminal experience.

Potential additions later:
- sample or preview generated inputs from `scripts/sample_inputs.py`,
- scale/convert external biological trace CSVs,
- inspect or compare config sidecars from saved runs,
- wrap small output-cleanup or export helpers.

Large workflows should stay in their dedicated notebooks or scripts rather than being absorbed here.
